# Radiology Explainer & Upload Demo — run entirely in Colab

This runs the **Snbig/rad_explain** space (a fork of `google/rad_explain`) **without any HF Space hosting**. It starts two local servers inside this Colab kernel:

* **MedGemma 1.5 chat-completions server** on `http://127.0.0.1:7861` (`serve_medgemma.py`)
* **rad_explain Flask app** on `http://127.0.0.1:7860` (report-explainer + **Upload** for X-ray / CT / MRI)

Then the app is embedded below as an interactive iframe. Everything (model included) runs on this Colab GPU.

> **Before starting:**
> 1. **Runtime → Change runtime type → T4 GPU** (via `/top` in Colab), then restart runtime.
> 2. Accept the model terms once: open [google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it) and click **Agree** (this is a per-account, not per-token, agreement).
> 3. Have a Hugging Face token with read access to gated models (Settings → [Access Tokens](https://huggingface.co/settings/tokens)).

## 1. Get the code

In [ ]:
import os, sys
REPO = "/content/rad_explain"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Snbig/rad_explain.git {REPO}
else:
    !git -C {REPO} pull --ff-only
sys.path.insert(0, REPO)
os.chdir(REPO)
print("repo ready:", REPO)


In [ ]:
# Install the app deps (flask, Pillow, diskcache, requests, numpy, pydicom)
# plus bitsandbytes so MedGemma loads in 4-bit and fits the T4.
!pip install -q -r requirements.txt bitsandbytes >/dev/null 2>&1
print("deps installed")


## 2. Authenticate to Hugging Face

In [ ]:
import os, getpass
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face token: ")
os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN set:", bool(HF_TOKEN))


## 3. Start MedGemma locally (first cell loads the ~4B model)

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# 4-bit (~2.5 GB) fits a T4. Use False only on a 24 GB+ GPU (L4).
LOAD_IN_4BIT = True  # @param {type:"boolean"}

import threading, socket, time
import serve_medgemma as sm

# Blocks until the model is downloaded + loaded; watch the progress bars.
sm.load_model(load_in_4bit=LOAD_IN_4BIT)
threading.Thread(target=lambda: sm.app.run(
    host="127.0.0.1", port=7861, threaded=True), daemon=True).start()

for _ in range(30):
    try:
        socket.create_connection(("127.0.0.1", 7861), timeout=1).close()
        break
    except OSError:
        time.sleep(1)
print("MedGemma server ready on http://127.0.0.1:7861")


In [ ]:
# Start the rad_explain Flask app pointed at the local MedGemma server.
import os, threading, socket, time
os.environ["MEDGEMMA_ENDPOINT_URL"] = "http://127.0.0.1:7861"
os.environ.setdefault("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
from app import app as flask_app
threading.Thread(target=lambda: flask_app.run(
    host="0.0.0.0", port=7860, threaded=True), daemon=True).start()
for _ in range(30):
    try:
        socket.create_connection(("127.0.0.1", 7860), timeout=1).close()
        break
    except OSError:
        time.sleep(1)
print("rad_explain app ready on http://127.0.0.1:7860")


## 4. Open the app

After running the cell below, the app appears embedded below (it may take a moment first time). Use the **Upload** tab to analyze your own X-ray / CT / MRI file, or pick one of the bundled demo reports and click a sentence.

To share a public URL from Colab you can also run:

```
# !pip install -q cloudflared && !cloudflared tunnel --url http://127.0.0.1:7860
```

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(7860, height=900)
except Exception as e:
    print("Could not open embedded iframe:", e)
    print("Open http://127.0.0.1:7860 in a browser tab (via the proxy in the Colab right panel).")


## Notes

* **First call after load can be slow** — generation is deterministic (`do_sample=False`, `temperature` ignored).
* **CT / MRI:** uploaded DICOM series are windowed and down-sampled to ≤30 slices server-side (the UI defaults to 8). On a 16 GB T4 keep the slice count low — if you hit CUDA OOM, upload fewer slices.
* **Modality:** auto-detected from DICOM tags (CT/MR/CR/DX/XC) with a manual override in the UI. MedGemma 1.5 natively interprets volumetric **CT and MRI** plus 2D **chest X-ray**.
* **Model caveats:** baseline capability only, not validated for clinical use; results are illustrative.
* Everything runs locally in this notebook — no HF Space, no paid endpoint. Your uploaded images never leave this runtime.